# Imports

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as mcm
from scipy import stats as scipy_stats
from scipy.stats import wasserstein_distance, ks_2samp, f_oneway
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titlesize': 11,
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 0.8,
    'axes.facecolor': '#f5f5f5',
    'figure.facecolor': 'white',
})

sys.path.insert(0, os.path.abspath('.'))
from partitioning import (
    IIDPartitioner,
    NoiseFeaturePartitioner,
    SyntheticFeaturePartitioner,
)

FEATURE_DIR = '../drc_prediction/training_set/DRC/feature/'
N_CHANNELS  = 9
CHANNEL_NAMES = [
    'macro_region', 'cell_density',
    'RUDY_long', 'RUDY_short', 'RUDY_pin_long',
    'congestion_eGR_H', 'congestion_eGR_V',
    'congestion_GR_H',  'congestion_GR_V',
]
print('Imports OK.')

# Filename parsing and metadata loading

In [ ]:
def parse_sample_name(filename: str) -> dict:
    """Parse a CircuitNet-N28 filename into its design-space components.

    Anchors from the right: the last 5 tokens always carry the prefixes
    c/u/m/p/f; the token at position -6 is macro_count; everything before
    that (after stripping any leading numeric id) is joined as the design name.

    Handles any number of hyphen-separated name parts, including:
      RISCY-a-1-c2-u0.7-m1-p1-f0.npy       → design=RISCY-a
      1-RISCY-a-1-c2-u0.7-m1-p1-f0.npy     → design=RISCY-a  (prefixed)
      zero-riscy-b-1-c2-u0.7-m1-p1-f0.npy  → design=zero-riscy-b
    """
    basename = filename.replace('.npy', '')
    parts = basename.split('-')
    # Strip leading numeric id if present (e.g. '1-RISCY-a-...')
    if parts[0].isdigit():
        parts = parts[1:]
    # Need at least: ≥1 design token + macro_count + c + u + m + p + f = 7
    if len(parts) < 7:
        raise ValueError(f'Cannot parse (too few tokens): {filename}')
    # Validate the 5 right-anchored suffix fields in order c, u, m, p, f
    for expected, token in zip(['c', 'u', 'm', 'p', 'f'], parts[-5:]):
        if not token.startswith(expected):
            raise ValueError(
                f'Cannot parse {filename}: expected prefix "{expected}", got "{token}"'
            )
    clock_str, util_str, macro_placement_raw, power_mesh_raw, filler_raw = parts[-5:]
    macro_count = parts[-6]
    design_name = '-'.join(parts[:-6])
    if not design_name:
        raise ValueError(f'Cannot parse {filename}: empty design name')
    return {
        'design_name':      design_name,
        'macro_count':      macro_count,
        'clock_ns':         float(clock_str[1:]),
        'utilization':      float(util_str[1:]),
        'macro_placement':  macro_placement_raw[1:],
        'power_mesh':       power_mesh_raw[1:],
        'filler_insertion': filler_raw[1:],
        'filename':         filename,
    }


feature_exists = os.path.isdir(FEATURE_DIR)
files = sorted(f for f in os.listdir(FEATURE_DIR) if f.endswith('.npy'))
print(f'Feature directory found: {len(files)} .npy files.')
records = []
for fname in files:
    try:
        records.append(parse_sample_name(fname))
    except Exception as e:
        print(f'  Skip {fname}: {e}')
df_meta = pd.DataFrame(records)

print(f'Samples: {len(df_meta)}')
print(df_meta.head())
print(df_meta.dtypes)

df_meta.loc[df_meta['macro_placement'].isin(['1','2','3','4']) == False, 'macro_placement'] = 1
df_meta['macro_placement'] = df_meta['macro_placement'].astype(int)

# Design Parameter Distributions

In [ ]:

CAT_COLS = ['design_name','macro_count','macro_placement','power_mesh','filler_insertion']
NUM_COLS = ['utilization','clock_ns']

print('=== Categorical parameter distributions ===')
for col in CAT_COLS:
    vc = df_meta[col].value_counts()
    balance = vc.min() / vc.max()
    print(f'\n{col}  (unique={vc.shape[0]}, balance={balance:.3f}):')
    print(vc.to_string())

print('\n=== Numerical parameter distributions ===')
for col in NUM_COLS:
    print(f'\n{col}:')
    print(df_meta[col].describe().to_string())
    print('  Unique values:', sorted(df_meta[col].unique()))

fig, axes = plt.subplots(3, 5, figsize=(24, 10))
fig.suptitle('Section A — Design Parameter Distributions', fontsize=14, fontweight='bold')

cmap10 = mcm.get_cmap('tab10')

for ax, col in zip(axes[0], CAT_COLS):
    vc = df_meta[col].value_counts().sort_index()
    colors = [cmap10(i % 10) for i in range(len(vc))]
    bars = ax.bar(range(len(vc)), vc.values, color=colors, alpha=0.85, edgecolor='white')
    ax.set_xticks(range(len(vc)))
    ax.set_xticklabels(vc.index, rotation=35, ha='right', fontsize=8)
    ax.set_title(col)
    ax.set_ylabel('Count')
    for bar, cnt in zip(bars, vc.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                str(cnt), ha='center', fontsize=7)

for ax, col in zip(axes[1], CAT_COLS):
    pivot2 = df_meta.groupby(['design_name', col]).size().unstack(fill_value=0)
    mat2 = pivot2.values.astype(float)
    im2 = ax.imshow(mat2, aspect='auto', cmap='Purples')
    ax.set_yticks(range(len(pivot2.index)))
    ax.set_yticklabels(pivot2.index, fontsize=8)
    ax.set_xticks(range(len(pivot2.columns)))
    ax.set_xticklabels([str(c) for c in pivot2.columns], rotation=30, ha='right', fontsize=7)
    ax.set_title(f'Sample count: design × {col}')
    plt.colorbar(im2, ax=ax, fraction=0.04)

# Utilization histogram
ax = axes[2][0]
ax.hist(df_meta['utilization'], bins=30, color=cmap10(5), alpha=0.85, edgecolor='white')
ax.set_title('utilization')
ax.set_xlabel('utilization ratio')
ax.set_ylabel('Count')

# Clock histogram
ax = axes[2][1]
ax.hist(df_meta['clock_ns'], bins=20, color=cmap10(6), alpha=0.85, edgecolor='white')
ax.set_title('clock_ns')
ax.set_xlabel('Clock period (ns)')
ax.set_ylabel('Count')

# Design × utilization sample counts heatmap
ax = axes[2][2]
util_bins = pd.cut(df_meta['utilization'], bins=5)
pivot = df_meta.groupby(['design_name', util_bins]).size().unstack(fill_value=0)
mat = pivot.values.astype(float)
im = ax.imshow(mat, aspect='auto', cmap='Blues')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=8)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns], rotation=30, ha='right', fontsize=7)
ax.set_title('Sample count: design × utilization bin')
plt.colorbar(im, ax=ax, fraction=0.04)

# Design × clock sample counts heatmap
ax = axes[2][3]
clock_bins = pd.cut(df_meta['clock_ns'], bins=5)
pivot2 = df_meta.groupby(['design_name', clock_bins]).size().unstack(fill_value=0)
mat2 = pivot2.values.astype(float)
im2 = ax.imshow(mat2, aspect='auto', cmap='Purples')
ax.set_yticks(range(len(pivot2.index)))
ax.set_yticklabels(pivot2.index, fontsize=8)
ax.set_xticks(range(len(pivot2.columns)))
ax.set_xticklabels([str(c) for c in pivot2.columns], rotation=30, ha='right', fontsize=7)
ax.set_title('Sample count: design × clock_ns bin')
plt.colorbar(im2, ax=ax, fraction=0.04)

plt.tight_layout()
plt.show()

# Design space coverage
all_cols = CAT_COLS + ['clock_ns', 'utilization']
unique_combos = df_meta[all_cols].drop_duplicates()
print(f'Unique parameter combinations: {len(unique_combos)} / {len(df_meta)} samples')